# Day 18 — 차등발현 결과 표를 만들고 읽기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-3/day-18.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-3/day-18.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — log2FC를 손으로 계산하기

In [ ]:
import numpy as np

flt = np.array([7.9, 8.1, 8.0])       # 어떤 유전자의 log2 발현값 (가상)
gc = np.array([9.0, 9.1, 8.9])

log2fc = flt.mean() - gc.mean()       # contrast: FLT - GC
print("log2FC:", round(log2fc, 2))
print("배수:", round(2 ** log2fc, 2), "(FLT가 GC의 몇 배인가)")

### 예시 2 — 아무 차이가 없어도 "유의한" 유전자는 나온다

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(1)
null_flt = rng.normal(8, 0.4, size=(6, 600))      # 6마리 × 600유전자, 군 차이 없음
null_gc = rng.normal(8, 0.4, size=(6, 600))

p = stats.ttest_ind(null_flt, null_gc, equal_var=False).pvalue
fdr = multipletests(p, method="fdr_bh")[1]
print("p < 0.05 인 유전자 수:", (p < 0.05).sum())
print("FDR < 0.05 인 유전자 수:", (fdr < 0.05).sum())

### 예시 3 — BH 보정이 p-value를 어떻게 바꾸는가

In [ ]:
p_small = np.array([0.001, 0.008, 0.039, 0.041, 0.20, 0.74])
adj = multipletests(p_small, method="fdr_bh")[1]
for a, b in zip(p_small, adj):
    print(f"p = {a:<6}  →  adjusted p = {b:.3f}")

## 직접 해보기

1. 설정 셀과 어제 저장한 표를 불러옵니다.

In [ ]:
# 이 셀은 Day 15–17의 산출물 파일이 필요합니다. Colab은 노트북마다 런타임이 따로라서,
# 그 파일을 이 런타임에 업로드하거나 Google Drive에 두고 경로를 맞추세요.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

MODE = "practice"          # 연습이 끝나면 "real"로 바꾸세요
RAW_DIR = Path("../data/osdr_536_practice" if MODE == "practice" else "../data/osdr_536")
OUT_DIR = RAW_DIR / "processed"
FIG_DIR = Path("../figures")

log2_norm = pd.read_csv(OUT_DIR / "log2_norm.csv", index_col=0)
gene_qc = pd.read_csv(OUT_DIR / "gene_qc.csv", index_col=0)
probes = pd.read_csv(OUT_DIR / "probes.csv")
group = pd.read_csv(OUT_DIR / "samples.csv").set_index("sample_id").loc[log2_norm.index, "group"]

keep = gene_qc.index[gene_qc["keep"]]
flt = log2_norm.loc[group == "FLT", keep]
gc = log2_norm.loc[group == "GC", keep]
print("FLT:", flt.shape, "| GC:", gc.shape)

2. 유전자마다 Welch t-test를 하고 표를 만듭니다. contrast는 FLT − GC입니다.

In [ ]:
res = stats.ttest_ind(flt, gc, equal_var=False)       # 열(유전자)마다 한 번씩

de = pd.DataFrame({
    "gene": keep,
    "mean_GC": gc.mean().values,
    "mean_FLT": flt.mean().values,
    "log2FC": (flt.mean() - gc.mean()).values,        # FLT - GC
    "p_value": res.pvalue,
})
de["FDR"] = multipletests(de["p_value"], method="fdr_bh")[1]
de = de.merge(probes[["Name", "Accession"]], left_on="gene", right_on="Name").drop(columns="Name")
de = de.sort_values("p_value").reset_index(drop=True)
print(de.head(8).round(4).to_string(index=False))

3. 표 전체를 숫자 몇 개로 요약합니다.

In [ ]:
n = len(de)
print("검정한 유전자 수:", n)
print("차이가 전혀 없어도 p < 0.05 로 기대되는 수:", round(n * 0.05, 1))
print("p < 0.05:", int((de["p_value"] < 0.05).sum()),
      "| 그중 FLT 감소:", int(((de["p_value"] < 0.05) & (de["log2FC"] < 0)).sum()))
print("FDR < 0.05:", int((de["FDR"] < 0.05).sum()), "| FDR < 0.10:", int((de["FDR"] < 0.10).sum()))
print("|log2FC| 최대:", round(de["log2FC"].abs().max(), 2))

4. 유전자 ID 체계를 확인합니다. `gene` 열은 유전자 기호(symbol), `Accession` 열은 전사체 번호입니다. 접두어별 개수를 세어 어떤 체계인지 봅니다.

In [ ]:
print(de["Accession"].str.split("_").str[0].value_counts().to_string())
print("기호가 중복된 유전자:", int(de["gene"].duplicated().sum()))

5. volcano plot으로 표 전체를 한 장에 봅니다. 가로축은 효과크기, 세로축은 −log10(p-value)입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sig = de["FDR"] < 0.05
ax.scatter(de.loc[~sig, "log2FC"], -np.log10(de.loc[~sig, "p_value"]), s=12, color="grey", label="FDR >= 0.05")
ax.scatter(de.loc[sig, "log2FC"], -np.log10(de.loc[sig, "p_value"]), s=18, color="tab:red", label="FDR < 0.05")
ax.axhline(-np.log10(0.05), linestyle="--", linewidth=0.8, color="black")
for _, row in de.head(8).iterrows():
    ax.annotate(row["gene"], (row["log2FC"], -np.log10(row["p_value"])), fontsize=7,
                xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("log2 fold change (FLT - GC)")
ax.set_ylabel("-log10(p-value)")
ax.set_title(f"Volcano plot (dashed line: unadjusted p = 0.05; color: FDR) [data: {MODE}]")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / f"osdr_{MODE}_volcano.png", dpi=150)

6. 전체 표와 상위 후보 표를 저장합니다. 상위 후보는 "p-value 순 상위 20개"로 정하되, 이 기준은 탐색용이라는 점을 열 이름으로도 남깁니다.

In [ ]:
de.to_csv(OUT_DIR / "de_table.csv", index=False)

top = de.head(20).copy()
top["direction"] = np.where(top["log2FC"] > 0, "FLT_up", "FLT_down")
top["passes_FDR_0.05"] = top["FDR"] < 0.05
top.round(4).to_csv(OUT_DIR / "top_candidates.csv", index=False)
print(top[["gene", "log2FC", "p_value", "FDR", "direction", "passes_FDR_0.05"]].head(5).round(4).to_string(index=False))